In [ ]:
"""in real world , how this will lead , give me exmaple siwth codes , how to us eit
In 2025, real-world Agentic AI is moving from simple chatbots to autonomous systems that perform multi-step tasks like automatic scheduling, fraud detection, and automated customer support. 
Instead of just answering a question, these agents plan a series of actions, execute them via tools (like APIs or databases), and then reflect on the results to decide if more work is needed. 
Real-World Example: "The Smart Researcher"
Imagine an agent that needs to answer: "What was the stock price of Apple yesterday, and how many shares can I buy with $5000?"
This requires two different tools: a Web Search to find the price and a Calculator to do the math.
1. Setup the Brain and Tools
python
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, create_react_agent

# Tool 1: Real-time search (Simulated)
@tool
def get_stock_price(symbol: str) -> float:
    Fetches current stock price for a given symbol.
    # Real world: This would call an API like AlphaVantage or Yahoo Finance
    prices = {"AAPL": 235.50, "GOOGL": 178.20}
    return prices.get(symbol.upper(), 0.0)

# Tool 2: Calculation
@tool
def calculator(expression: str) -> str:
    Evaluates mathematical expressions.
    return str(eval(expression))

tools = [get_stock_price, calculator]
llm = ChatOpenAI(model="gpt-4o-2025-01-01") # Using 2025 state-of-the-art model
Use code with caution.

2. Create the Agent Workflow
In the real world, we use create_react_agent for a quick, robust setup that includes the "Reason + Act" loop automatically. 
python
# The agent is built with a "System Prompt" to give it a personality/instructions
system_message = "You are a financial research assistant. Use tools to find facts before calculating."

agent_executor = create_react_agent(llm, tools, state_modifier=system_message)
Use code with caution.

3. How to Run (Testing the Loop)
When you run this, the agent doesn't just guess. It follows these steps:
Reason: "I need Apple's price first." → Act: Calls get_stock_price("AAPL").
Observe: Receives 235.50.
Reason: "Now I need to divide 5000 by 235.50." → Act: Calls calculator("5000 / 235.50").
Observe: Receives 21.23.
Final Answer: "You can buy approximately 21 shares."
python
response = agent_executor.invoke({
    "messages": [("user", "How many Apple (AAPL) shares can I buy for $5000?")]
})

print(response["messages"][-1].content)
Use code with caution.

Real-World Use Case Scenarios
Customer Support: An agent receives a complaint, searches the database for the user's order ID using a database_tool, and then issues a refund via a payment_api_tool.
IT Automation: An agent monitors server logs; if it sees an error, it uses a terminal_tool to restart the service and a slack_tool to notify the team.
Healthcare Assistants: Agents summarize patient history by calling a records_tool and suggest potential diagnoses based on current symptoms. 
Key Benefits of this Approach
Reliability: The agent checks facts rather than hallucinating.
Persistence: LangGraph can "save" the state. If the calculator fails, the agent can try again without losing the stock price it already found.
Human-in-the-Loop: In 2025, real-world apps often add an interrupt so a human must approve a "Buy" or "Refund" action before the tool actually executes. 



"""

In [1]:
# Install dependencies
!pip install langchain-openai langgraph langchain-core


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Imports
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

c:\Users\USER\Desktop\pinecone-demo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Setup API credentials (A4F API)
os.environ['OPENAI_API_KEY'] = 'ddc-a4f-979e772e101449daa8ebf58e4e3be7ad'
os.environ['OPENAI_API_BASE'] = 'https://api.a4f.co/v1'

In [4]:
# Define Tools

@tool
def get_stock_price(symbol: str) -> float:
    """Fetches current stock price for a given symbol."""
    # Real world: This would call an API like AlphaVantage or Yahoo Finance
    prices = {"AAPL": 235.50, "GOOGL": 178.20, "MSFT": 425.80, "TSLA": 248.50, "AMZN": 198.75}
    return prices.get(symbol.upper(), 0.0)

@tool
def calculator(expression: str) -> str:
    """Evaluates mathematical expressions safely."""
    try:
        # Only allow safe math operations
        allowed_chars = set('0123456789+-*/.() ')
        if all(c in allowed_chars for c in expression):
            return str(eval(expression))
        else:
            return "Error: Invalid expression"
    except Exception as e:
        return f"Error: {str(e)}"

tools = [get_stock_price, calculator]
print("✅ Tools defined:", [t.name for t in tools])

✅ Tools defined: ['get_stock_price', 'calculator']


In [5]:
# Initialize LLM with A4F API (provider-3 supports tool calling)
llm = ChatOpenAI(
    model="provider-3/openai/gpt-4o",  # provider-3 supports tool/function calling
    temperature=0,
    api_key=os.environ['OPENAI_API_KEY'],
    base_url="https://api.a4f.co/v1"
)
print("✅ LLM initialized:", llm.model_name)

✅ LLM initialized: provider-3/openai/gpt-4o


In [6]:
# Create the ReAct Agent
system_message = """You are a financial research assistant. 
Use tools to find facts before calculating. 
Always use the get_stock_price tool first to get accurate prices, 
then use the calculator tool for any math operations."""

agent_executor = create_react_agent(llm, tools, state_modifier=system_message)
print("✅ Agent created successfully!")

C:\Users\USER\AppData\Local\Temp\ipykernel_10164\2579439610.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools, state_modifier=system_message)


TypeError: create_react_agent() got unexpected keyword arguments: {'state_modifier': 'You are a financial research assistant. \nUse tools to find facts before calculating. \nAlways use the get_stock_price tool first to get accurate prices, \nthen use the calculator tool for any math operations.'}

## 🧪 Test the Agent

The agent will follow these steps:
1. **Reason**: "I need Apple's price first." → **Act**: Calls `get_stock_price("AAPL")`
2. **Observe**: Receives 235.50
3. **Reason**: "Now I need to divide 5000 by 235.50." → **Act**: Calls `calculator("5000 / 235.50")`
4. **Observe**: Receives 21.23
5. **Final Answer**: "You can buy approximately 21 shares."

In [ ]:
# Test 1: How many shares can I buy?
response = agent_executor.invoke({
    "messages": [("user", "How many Apple (AAPL) shares can I buy for $5000?")]
})

print("🎯 Final Answer:")
print(response["messages"][-1].content)

In [ ]:
# See the full reasoning chain
print("=== 🔍 Full Agent Reasoning Chain ===\n")
for i, msg in enumerate(response["messages"]):
    print(f"Step {i+1} - {msg.type.upper()}:")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  🔧 Tool Call: {tc['name']}({tc['args']})")
    elif hasattr(msg, 'content') and msg.content:
        content = str(msg.content)
        print(f"  💬 {content[:300]}..." if len(content) > 300 else f"  💬 {content}")
    print()

In [ ]:
# Test 2: Compare stock prices
response2 = agent_executor.invoke({
    "messages": [("user", "What is the price of Google (GOOGL) stock and how much would 50 shares cost?")]
})

print("🎯 Final Answer:")
print(response2["messages"][-1].content)

In [ ]:
# Test 3: Complex multi-step query
response3 = agent_executor.invoke({
    "messages": [("user", "I have $10000. Compare how many shares of Microsoft (MSFT) vs Tesla (TSLA) I can buy.")]
})

print("🎯 Final Answer:")
print(response3["messages"][-1].content)

## 🌍 Real-World Use Case Scenarios

| Use Case | Tools Used | Example |
|----------|------------|---------|
| **Customer Support** | `database_tool`, `payment_api_tool` | Agent finds order ID, issues refund |
| **IT Automation** | `terminal_tool`, `slack_tool` | Agent restarts failed service, notifies team |
| **Healthcare** | `records_tool`, `diagnosis_tool` | Agent summarizes patient history, suggests diagnosis |
| **Finance** | `stock_api_tool`, `calculator` | Agent researches prices, calculates portfolio |

## ✅ Key Benefits
- **Reliability**: Agent checks facts via tools instead of hallucinating
- **Persistence**: LangGraph saves state - if calculator fails, agent retries without losing previous data
- **Human-in-the-Loop**: Add interrupts for human approval before sensitive actions (Buy, Refund, Delete)